# Demo 02 — DDIM Fast Sampling + Noise Schedule Ablation
### TLH-1 · Applied Models: Diffusion Models

**Prerequisite:** Run `demo_01_ddpm_mnist.ipynb` first — this notebook loads the trained model.

**What we explore:**
1. DDIM deterministic ODE sampling — same model, 50 steps instead of 1000
2. Speed/quality trade-off: 1000 vs 200 vs 50 vs 10 steps
3. Noise schedule ablation: linear vs cosine schedule
4. Latent space interpolation: walk between two noise seeds

---

**The key DDIM insight** (Song et al. 2020):

The DDPM training objective only depends on $q(x_t | x_0)$ — the marginals — not the full Markov chain.
So we can define a **different (non-Markovian) reverse process** with the same marginals but fewer steps.

The result is a deterministic ODE:
$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \underbrace{\frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}}_{\hat x_0} + \sqrt{1-\bar\alpha_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Re-define everything from Demo 01 ────────────────────────────────────────
T          = 1000
beta_start = 1e-4
beta_end   = 0.02
img_size   = 28
batch_size = 128
lr         = 2e-4
epochs     = 10
model_dim  = 64

betas     = torch.linspace(beta_start, beta_end, T, device=device)
alphas    = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)
sqrt_ab   = alpha_bar.sqrt()
sqrt_1mab = (1.0 - alpha_bar).sqrt()

def extract(a, t, x_shape):
    b   = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    return extract(sqrt_ab, t, x0.shape) * x0 + extract(sqrt_1mab, t, x0.shape) * noise

def sinusoidal_embedding(t, dim):
    half  = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
    args  = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1  = nn.GroupNorm(8, in_ch)
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.norm2  = nn.GroupNorm(8, out_ch)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act    = nn.SiLU()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = h + self.t_proj(self.act(t_emb))[:, :, None, None]
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.res  = ResBlock(in_ch, out_ch, t_dim)
        self.down = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)
    def forward(self, x, t_emb):
        x = self.res(x, t_emb)
        return self.down(x), x

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, t_dim):
        super().__init__()
        self.up  = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.res = ResBlock(in_ch + skip_ch, out_ch, t_dim)
    def forward(self, x, skip, t_emb):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x, t_emb)

class UNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, t_dim=256):
        super().__init__()
        self.t_mlp      = nn.Sequential(nn.Linear(base_ch, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.init_conv  = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.down1      = DownBlock(base_ch,     base_ch * 2, t_dim)
        self.down2      = DownBlock(base_ch * 2, base_ch * 4, t_dim)
        self.bottleneck = ResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.up1        = UpBlock(base_ch * 4, base_ch * 2, base_ch * 2, t_dim)
        self.up2        = UpBlock(base_ch * 2, base_ch,     base_ch,     t_dim)
        self.out_conv   = nn.Conv2d(base_ch, in_ch, 1)
    def forward(self, x, t):
        t_emb  = self.t_mlp(sinusoidal_embedding(t, model_dim))
        x      = self.init_conv(x)
        x, s1  = self.down1(x, t_emb)
        x, s2  = self.down2(x, t_emb)
        x      = self.bottleneck(x, t_emb)
        x      = self.up1(x, s2, t_emb)
        x      = self.up2(x, s1, t_emb)
        return self.out_conv(x)

model     = UNet(in_ch=1, base_ch=model_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print('Model defined. Training for', epochs, 'epochs...')

In [ ]:
# ── Quick training (same as Demo 01 — skip if you saved weights) ─────────────
dataset = datasets.MNIST(
    root='./data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2 - 1)
    ])
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

loss_history = []
for epoch in range(1, epochs + 1):
    model.train()
    ep_loss = 0.0
    for x0, _ in tqdm(loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
        x0   = x0.to(device)
        t    = torch.randint(0, T, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        x_t  = q_sample(x0, t, noise)
        pred = model(x_t, t)
        loss = F.mse_loss(pred, noise)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    avg = ep_loss / len(loader)
    loss_history.append(avg)
    print(f'Epoch {epoch:2d} | loss {avg:.4f}')

## Part 1: DDIM Sampler

DDIM uses a subset of $S$ timesteps from the full $T=1000$ schedule. At each step:

1. Predict the clean image: $\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta}{\sqrt{\bar\alpha_t}}$
2. Re-noise to the next (earlier) level: $x_{t-1} = \sqrt{\bar\alpha_{t-1}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t-1}}\,\epsilon_\theta$

This is the deterministic (η=0) DDIM update.

In [ ]:
@torch.no_grad()
def p_sample_ddim(x_T, n_steps=50):
    """
    DDIM deterministic sampling (eta=0).
    n_steps: number of denoising steps (subset of T=1000 timesteps).
    """
    model.eval()
    n_samples = x_T.shape[0]

    # Choose n_steps evenly-spaced timesteps from [0, T-1]
    step_seq  = torch.linspace(0, T - 1, n_steps + 1, dtype=torch.long)
    step_seq  = list(reversed(step_seq.tolist()))   # go from T → 0

    x = x_T
    for i in tqdm(range(len(step_seq) - 1), desc=f'DDIM ({n_steps} steps)', leave=False):
        t_curr = step_seq[i]
        t_prev = step_seq[i + 1]

        t_curr_t = torch.full((n_samples,), t_curr, device=device, dtype=torch.long)

        eps_pred = model(x, t_curr_t)

        ab_curr  = alpha_bar[t_curr]
        ab_prev  = alpha_bar[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=device)

        # Predict x_0 from x_t
        x0_pred  = (x - (1 - ab_curr).sqrt() * eps_pred) / ab_curr.sqrt()
        x0_pred  = x0_pred.clamp(-1, 1)   # stabilise

        # DDIM update — deterministic (eta=0)
        x = ab_prev.sqrt() * x0_pred + (1 - ab_prev).sqrt() * eps_pred

    return x

print('DDIM sampler defined.')

## Part 2: Speed vs Quality — 1000 vs 200 vs 50 vs 10 Steps

In [ ]:
@torch.no_grad()
def p_sample_ddpm_fast(x_T):
    """DDPM sampling — 1000 steps (reproduced from Demo 01)."""
    x = x_T.clone()
    model.eval()
    for t_val in tqdm(reversed(range(T)), total=T, desc='DDPM 1000 steps', leave=False):
        t_batch  = torch.full((x.shape[0],), t_val, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        alpha_t  = extract(alphas,    t_batch, x.shape)
        ab_t     = extract(alpha_bar, t_batch, x.shape)
        beta_t   = extract(betas,     t_batch, x.shape)
        mu = (1.0 / alpha_t.sqrt()) * (x - beta_t / (1 - ab_t).sqrt() * eps_pred)
        x  = mu + (beta_t.sqrt() * torch.randn_like(x) if t_val > 0 else 0)
    return x


n_samples  = 16
x_T_fixed  = torch.randn(n_samples, 1, img_size, img_size, device=device)

configs = [
    ('DDPM 1000',  lambda: p_sample_ddpm_fast(x_T_fixed)),
    ('DDIM 200',   lambda: p_sample_ddim(x_T_fixed, 200)),
    ('DDIM 50',    lambda: p_sample_ddim(x_T_fixed, 50)),
    ('DDIM 10',    lambda: p_sample_ddim(x_T_fixed, 10)),
]

results  = {}
timings  = {}
for name, fn in configs:
    t0           = time.time()
    results[name] = fn()
    timings[name] = time.time() - t0
    print(f'{name:15s}  {timings[name]:.1f}s')

# ── Side-by-side grid ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(configs), 8, figsize=(12, len(configs) * 1.6))
for row, (name, _) in enumerate(configs):
    samples = results[name]
    for col in range(8):
        img = samples[col, 0].cpu().clamp(-1, 1).numpy()
        axes[row, col].imshow((img + 1) / 2, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{name}\n({timings[name]:.1f}s)', fontsize=8)

plt.suptitle('Same model — varying number of sampling steps', y=1.01)
plt.tight_layout()
plt.show()

## Part 3: Noise Schedule Ablation — Linear vs Cosine

Nichol & Dhariwal (2021) showed the **cosine schedule** gives a more uniform noise level across $t$:

$$\bar\alpha_t = \cos\!\left(\frac{t/T + s}{1 + s} \cdot \frac{\pi}{2}\right)^2$$

The linear schedule destroys too much information at low $t$ and too little at high $t$.

In [ ]:
def cosine_schedule(T, s=0.008):
    """Cosine noise schedule (Nichol & Dhariwal 2021)."""
    steps     = torch.arange(T + 1, dtype=torch.float32) / T
    f         = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    alpha_bar = f / f[0]                              # normalise so ᾱ_0 = 1
    betas     = 1 - alpha_bar[1:] / alpha_bar[:-1]   # recover β from ᾱ
    betas     = betas.clamp(max=0.999)
    return betas, alpha_bar[1:]


betas_cos, alpha_bar_cos = cosine_schedule(T)
betas_lin                = torch.linspace(beta_start, beta_end, T)
alpha_bar_lin            = torch.cumprod(1.0 - betas_lin, dim=0)

# ── Compare schedules ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(alpha_bar_lin, label='Linear', color='#8f3e22', linewidth=2)
axes[0].plot(alpha_bar_cos, label='Cosine', color='#2a5c8a', linewidth=2, linestyle='--')
axes[0].set_xlabel('timestep t'); axes[0].set_ylabel('ᾱ_t')
axes[0].set_title('Signal remaining vs timestep'); axes[0].legend(); axes[0].grid(alpha=0.3)

betas_lin_np = betas_lin.numpy()
betas_cos_np = betas_cos.numpy()
axes[1].plot(betas_lin_np, label='Linear β', color='#8f3e22', linewidth=2)
axes[1].plot(betas_cos_np, label='Cosine β', color='#2a5c8a', linewidth=2, linestyle='--')
axes[1].set_xlabel('timestep t'); axes[1].set_ylabel('β_t')
axes[1].set_title('Noise added per step'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Linear vs Cosine noise schedule comparison', y=1.02)
plt.tight_layout()
plt.show()

print('Cosine: first ᾱ drops more slowly → preserves signal longer at small t')
print('Linear: last ᾱ near-zero too early → most steps near pure noise')

In [ ]:
# Visualise how each schedule corrupts an image
dataset = datasets.MNIST(
    root='./data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2 - 1)
    ])
)
x0_demo, _ = next(iter(DataLoader(dataset, batch_size=4, shuffle=True)))
x0_demo    = x0_demo[:1].to(device)

tsteps     = [0, 100, 250, 500, 750, 999]
fig, axes  = plt.subplots(2, len(tsteps), figsize=(12, 4))

for col, t_val in enumerate(tsteps):
    t_tensor = torch.full((1,), t_val, dtype=torch.long)

    # Linear schedule corruption
    ab_l   = alpha_bar_lin[t_val]
    noise  = torch.randn_like(x0_demo.cpu())
    x_lin  = ab_l.sqrt() * x0_demo.cpu() + (1 - ab_l).sqrt() * noise
    axes[0, col].imshow((x_lin[0, 0].clamp(-1,1).numpy() + 1) / 2, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f't={t_val}', fontsize=8)
    axes[0, col].axis('off')

    # Cosine schedule corruption
    ab_c   = alpha_bar_cos[t_val]
    x_cos  = ab_c.sqrt() * x0_demo.cpu() + (1 - ab_c).sqrt() * noise
    axes[1, col].imshow((x_cos[0, 0].clamp(-1,1).numpy() + 1) / 2, cmap='gray', vmin=0, vmax=1)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Linear', fontsize=9)
axes[1, 0].set_ylabel('Cosine', fontsize=9)
plt.suptitle('Image corruption: linear vs cosine schedule', y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Latent Space Interpolation

Because DDIM is **deterministic**, the same noise seed always produces the same image.
We can smoothly interpolate between two noise vectors $z_1$ and $z_2$ in the latent space.

In [ ]:
@torch.no_grad()
def slerp(z1, z2, t):
    """Spherical linear interpolation between two noise vectors."""
    z1_flat = z1.view(1, -1)
    z2_flat = z2.view(1, -1)
    norm1   = z1_flat.norm(dim=1, keepdim=True)
    norm2   = z2_flat.norm(dim=1, keepdim=True)
    z1_n    = z1_flat / norm1
    z2_n    = z2_flat / norm2
    dot     = (z1_n * z2_n).sum(dim=1, keepdim=True).clamp(-1, 1)
    theta   = dot.acos()
    if theta.abs() < 1e-6:
        return (1 - t) * z1 + t * z2
    interp  = (torch.sin((1 - t) * theta) * z1_flat + torch.sin(t * theta) * z2_flat) / theta.sin()
    return interp.view_as(z1)


torch.manual_seed(0)
z1 = torch.randn(1, 1, img_size, img_size, device=device)
torch.manual_seed(7)
z2 = torch.randn(1, 1, img_size, img_size, device=device)

n_interp = 10
interp_samples = []
for i, alpha in enumerate(torch.linspace(0, 1, n_interp)):
    z_interp        = slerp(z1, z2, alpha.item())
    sample          = p_sample_ddim(z_interp, n_steps=50)
    interp_samples.append(sample[0, 0].cpu().clamp(-1, 1).numpy())

fig, axes = plt.subplots(1, n_interp, figsize=(14, 2))
for i, (ax, img) in enumerate(zip(axes, interp_samples)):
    ax.imshow((img + 1) / 2, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{i/(n_interp-1):.1f}', fontsize=7)
    ax.axis('off')
plt.suptitle('Spherical latent interpolation (z₁ → z₂) via DDIM 50 steps', y=1.05)
plt.tight_layout()
plt.show()

## Summary

| Sampler | Steps | Time (CPU, 16 samples) | Quality |
|---|---|---|---|
| DDPM | 1000 | ~45s | Best |
| DDIM | 200 | ~9s | ≈ DDPM |
| DDIM | 50 | ~2.5s | Very close |
| DDIM | 10 | ~0.5s | Acceptable |

**Key insight:** DDIM is not a new model — it's a new *sampler* for the same trained weights.
The probability flow ODE formulation is what unlocks determinism and step-skipping.

---

**Next session (TLH-2):** Flow Matching — what if the forward process was a straight line?
- Simpler math (linear ODE paths, no SDE)
- Faster training
- Same quality (powers Stable Diffusion 3, FLUX)
- Implement rectified flows on MNIST